# Real-Time Face Detection with YOLO26m (Notebook-First, End-to-End)

## 1. Title and Project Overview

This notebook is a real, executable face detection workflow trained on the WIDER FACE benchmark.

What this notebook does:
- downloads the `mksaad/wider-face-a-face-detection-benchmark` dataset from Kaggle in notebook cells
- converts WIDER FACE annotations to YOLO format
- verifies images, labels, and splits before training
- trains a **YOLO26m** face detector (fallback to YOLO26s on OOM)
- evaluates with mAP50, mAP50-95, precision, recall, and per-class AP
- saves `best.pt`, `best.onnx`, and `metrics.json`

## 2. Problem Statement

Detect and localise all human faces in images with bounding boxes.

- Input: RGB images (portraits, street scenes, group photos, surveillance)
- Output: bounding boxes + confidence for class `face`
- Dataset: WIDER FACE — 32,203 images, 393,703 labelled face bboxes across 61 event categories
- Challenges: extreme scale variation (tiny faces to close-up portraits), occlusion, blur, illumination

## 3. Why the Chosen Method Is Correct

**Task family:** object detection (face-specialised).

- Bounding-box localisation is required — classification cannot locate faces
- Per CV rules: face detection uses the YOLO26m face detector pipeline
- YOLO26m is the April 2026 default for trainable single-class detection; face detection is a subcase
- Fallback to YOLO26s only on real GPU OOM

## 4. Hardware / Environment Check

In [ ]:
import os, platform, random
import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Python  : {platform.python_version()}")
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()} — {torch.version.cuda if torch.cuda.is_available() else 'N/A'}")
print(f"Device  : {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
    print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

## 5. Dependency Installation

In [ ]:
import subprocess, sys, importlib

def ensure_package(import_name: str, pip_name: str | None = None) -> None:
    pip_name = pip_name or import_name
    try:
        importlib.import_module(import_name)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])

ensure_package("ultralytics")
ensure_package("cv2", "opencv-python")
ensure_package("matplotlib")
ensure_package("pandas")
ensure_package("kagglehub")
print("All dependencies satisfied.")

## 6. Imports and Configuration

In [ ]:
import json, os, shutil, random
from pathlib import Path

import cv2
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

PROJECT_DIR   = Path(os.path.dirname(os.path.abspath("__file__")))
DATA_ROOT     = PROJECT_DIR.parents[2] / "data"
RUNS_DIR      = PROJECT_DIR.parents[2] / "runs"
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"

for d in (DATA_ROOT, RUNS_DIR, ARTIFACTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

print(f"DATA_ROOT    : {DATA_ROOT}")
print(f"ARTIFACTS_DIR: {ARTIFACTS_DIR}")

## 7. Dataset Source Explanation

**Dataset:** Kaggle — `mksaad/wider-face-a-face-detection-benchmark`

- Source URL: https://www.kaggle.com/datasets/mksaad/wider-face-a-face-detection-benchmark
- WIDER FACE: 32,203 images with 393,703 annotated faces
- Official train/val splits with `wider_face_split/` annotation `.mat` files
- Format: MATLAB `.mat` annotation files → converted to YOLO format by this notebook

**Credential requirements:**
- Set `KAGGLE_USERNAME` + `KAGGLE_KEY` env vars, or place `kaggle.json` in `~/.kaggle/`
- Raises a clear error if credentials are missing — no synthetic fallback

In [ ]:
import subprocess

KAGGLE_DATASET = "mksaad/wider-face-a-face-detection-benchmark"
DATASET_DIR = DATA_ROOT / "wider_face"
DATASET_DIR.mkdir(parents=True, exist_ok=True)


def check_kaggle_credentials() -> None:
    has_env = os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY")
    has_file = (Path.home() / ".kaggle" / "kaggle.json").exists()
    if not has_env and not has_file:
        raise RuntimeError(
            "Kaggle credentials not found.\n"
            "Set KAGGLE_USERNAME and KAGGLE_KEY env vars, or place kaggle.json in ~/.kaggle/"
        )


def download_dataset() -> Path:
    check_kaggle_credentials()
    try:
        import kagglehub
        path = Path(kagglehub.dataset_download(KAGGLE_DATASET))
        return path
    except Exception:
        pass
    subprocess.check_call([
        "kaggle", "datasets", "download", "-d", KAGGLE_DATASET,
        "-p", str(DATASET_DIR), "--unzip",
    ])
    return DATASET_DIR


print("Downloading WIDER FACE dataset from Kaggle...")
raw_root = download_dataset()
print(f"Raw dataset at: {raw_root}")

# Discover folder layout
print("\nTop-level contents:")
for p in sorted(Path(raw_root).iterdir()):
    print(f"  {p.name}{'/' if p.is_dir() else ''}")

## 8. Annotation Conversion — WIDER FACE → YOLO Format

WIDER FACE annotations are stored in MATLAB `.mat` files.
This cell parses them and writes YOLO-format `.txt` label files.

In [ ]:
import scipy.io

def find_wider_face_root(raw_root: Path) -> Path:
    """Find the root that contains WIDER_train, WIDER_val image folders."""
    for candidate in [raw_root, *raw_root.iterdir()]:
        if candidate.is_dir():
            if (candidate / "WIDER_train").exists() or (candidate / "WIDER_val").exists():
                return candidate
            # one level deeper
            for sub in candidate.iterdir():
                if sub.is_dir() and ((sub / "WIDER_train").exists() or (sub / "WIDER_val").exists()):
                    return sub
    raise RuntimeError(f"Could not find WIDER_train / WIDER_val under {raw_root}")


wider_root = find_wider_face_root(Path(raw_root))
print(f"WIDER FACE root: {wider_root}")

# Locate annotation mat files
mat_candidates = list(Path(raw_root).rglob("wider_face_train_bbx_gt.mat"))
if not mat_candidates:
    mat_candidates = list(Path(raw_root).rglob("*train*.mat"))
if not mat_candidates:
    raise RuntimeError("Could not find WIDER FACE annotation .mat file under download root.")

mat_train = mat_candidates[0]
print(f"Train annotation: {mat_train}")

mat_val_candidates = list(Path(raw_root).rglob("wider_face_val_bbx_gt.mat"))
if not mat_val_candidates:
    mat_val_candidates = list(Path(raw_root).rglob("*val*.mat"))
mat_val = mat_val_candidates[0] if mat_val_candidates else None
print(f"Val annotation  : {mat_val}")


def parse_wider_mat(mat_path: Path):
    """Parse WIDER FACE .mat file; return list of (event, filename, bboxes_nx4)."""
    data = scipy.io.loadmat(str(mat_path), squeeze_me=True, struct_as_record=False)
    records = []
    try:
        event_list = data["event_list"]
        file_list  = data["file_list"]
        face_bbx   = data["face_bbx_list"]
        for i, event in enumerate(event_list):
            event_name = str(event).strip()
            files = file_list[i]
            bboxes_list = face_bbx[i]
            # Handle scalar vs array
            if not hasattr(files, "__len__"):
                files = [files]
                bboxes_list = [bboxes_list]
            for j, fname in enumerate(files):
                fname_str = str(fname).strip()
                bboxes = bboxes_list[j]
                if hasattr(bboxes, "shape") and bboxes.ndim == 1 and bboxes.shape[0] == 4:
                    bboxes = bboxes.reshape(1, 4)
                elif not hasattr(bboxes, "shape"):
                    bboxes = np.array(bboxes).reshape(-1, 4)
                records.append((event_name, fname_str, bboxes))
    except KeyError as e:
        raise RuntimeError(f"Unexpected .mat structure: {e}")
    return records


print("Parsing train annotations...")
train_records = parse_wider_mat(mat_train)
print(f"  Train records: {len(train_records):,}")

val_records = []
if mat_val:
    print("Parsing val annotations...")
    val_records = parse_wider_mat(mat_val)
    print(f"  Val records  : {len(val_records):,}")

In [ ]:
def convert_to_yolo(records, split_name: str, wider_root: Path, yolo_root: Path,
                    max_images: int = 10000) -> tuple[int, int]:
    """Convert WIDER FACE records → YOLO dataset; return (n_ok, n_skip)."""
    img_out = yolo_root / "images" / split_name
    lbl_out = yolo_root / "labels" / split_name
    img_out.mkdir(parents=True, exist_ok=True)
    lbl_out.mkdir(parents=True, exist_ok=True)

    src_img_root = wider_root / f"WIDER_{split_name}"
    if not src_img_root.exists():
        # Try alternate naming (WIDER_train → WIDER_train/images)
        alt = wider_root / f"WIDER_{split_name}" / "images"
        if alt.exists():
            src_img_root = alt

    n_ok, n_skip, n_total = 0, 0, 0
    for event_name, fname, bboxes in records:
        if n_total >= max_images:
            break
        n_total += 1

        # locate source image
        img_path = src_img_root / "images" / event_name / f"{fname}.jpg"
        if not img_path.exists():
            img_path = src_img_root / event_name / f"{fname}.jpg"
        if not img_path.exists():
            n_skip += 1
            continue

        img = cv2.imread(str(img_path))
        if img is None:
            n_skip += 1
            continue
        h, w = img.shape[:2]

        # Convert each bbox [x, y, bw, bh] (WIDER: top-left corner + size)
        yolo_lines = []
        for bbox in bboxes:
            if len(bbox) < 4:
                continue
            x, y, bw, bh = float(bbox[0]), float(bbox[1]), float(bbox[2]), float(bbox[3])
            if bw <= 0 or bh <= 0:
                continue
            xc = (x + bw / 2) / w
            yc = (y + bh / 2) / h
            nw = bw / w
            nh = bh / h
            xc  = max(0.0, min(1.0, xc))
            yc  = max(0.0, min(1.0, yc))
            nw  = max(0.0, min(1.0, nw))
            nh  = max(0.0, min(1.0, nh))
            yolo_lines.append(f"0 {xc:.6f} {yc:.6f} {nw:.6f} {nh:.6f}")

        # Write image and label
        dst_img = img_out / f"{event_name}_{fname}.jpg"
        dst_lbl = lbl_out / f"{event_name}_{fname}.txt"
        shutil.copy2(img_path, dst_img)
        dst_lbl.write_text("\n".join(yolo_lines))
        n_ok += 1

    return n_ok, n_skip


# Cap at 8000 train + 2000 val to keep training tractable
MAX_TRAIN = 8000
MAX_VAL   = 2000

YOLO_ROOT = DATASET_DIR / "yolo_dataset"

print(f"Converting train (up to {MAX_TRAIN} images)...")
train_ok, train_skip = convert_to_yolo(train_records, "train", wider_root, YOLO_ROOT, MAX_TRAIN)
print(f"  OK: {train_ok:,}  Skipped: {train_skip:,}")

if val_records:
    print(f"Converting val (up to {MAX_VAL} images)...")
    val_ok, val_skip = convert_to_yolo(val_records, "val", wider_root, YOLO_ROOT, MAX_VAL)
    print(f"  OK: {val_ok:,}  Skipped: {val_skip:,}")
else:
    # Create a val split from train
    print("No val annotations found — carving 15% val from train images...")
    train_imgs = list((YOLO_ROOT / "images" / "train").glob("*.jpg"))
    random.seed(SEED)
    random.shuffle(train_imgs)
    val_cut = int(0.15 * len(train_imgs))
    val_imgs_to_move = train_imgs[:val_cut]
    (YOLO_ROOT / "images" / "val").mkdir(parents=True, exist_ok=True)
    (YOLO_ROOT / "labels" / "val").mkdir(parents=True, exist_ok=True)
    for img_p in val_imgs_to_move:
        lbl_p = YOLO_ROOT / "labels" / "train" / img_p.with_suffix(".txt").name
        shutil.move(str(img_p), YOLO_ROOT / "images" / "val" / img_p.name)
        if lbl_p.exists():
            shutil.move(str(lbl_p), YOLO_ROOT / "labels" / "val" / lbl_p.name)
    print(f"  Moved {len(val_imgs_to_move)} images to val split.")

# Create a test split from remaining train
train_imgs_remaining = list((YOLO_ROOT / "images" / "train").glob("*.jpg"))
random.seed(SEED)
random.shuffle(train_imgs_remaining)
test_cut = int(0.10 * len(train_imgs_remaining))
test_imgs_to_move = train_imgs_remaining[:test_cut]
(YOLO_ROOT / "images" / "test").mkdir(parents=True, exist_ok=True)
(YOLO_ROOT / "labels" / "test").mkdir(parents=True, exist_ok=True)
for img_p in test_imgs_to_move:
    lbl_p = YOLO_ROOT / "labels" / "train" / img_p.with_suffix(".txt").name
    shutil.move(str(img_p), YOLO_ROOT / "images" / "test" / img_p.name)
    if lbl_p.exists():
        shutil.move(str(lbl_p), YOLO_ROOT / "labels" / "test" / lbl_p.name)
print(f"  Carved {len(test_imgs_to_move)} images into test split.")

# Write data.yaml
DATA_YAML = YOLO_ROOT / "data.yaml"
DATA_YAML.write_text(
    f"path: {YOLO_ROOT}\n"
    "train: images/train\n"
    "val: images/val\n"
    "test: images/test\n"
    "nc: 1\n"
    "names: ['face']\n"
)
print(f"\ndata.yaml written: {DATA_YAML}")

## 9. Dataset Verification

Fail loudly if expected files are missing.

In [ ]:
def count_split(split_name: str) -> tuple[int, int]:
    img_dir = YOLO_ROOT / "images" / split_name
    lbl_dir = YOLO_ROOT / "labels" / split_name
    imgs = list(img_dir.glob("*.jpg")) if img_dir.exists() else []
    lbls = list(lbl_dir.glob("*.txt")) if lbl_dir.exists() else []
    return len(imgs), len(lbls)

ti, tl = count_split("train")
vi, vl = count_split("val")
sti, stl = count_split("test")

print(f"Train : {ti:5d} images, {tl:5d} labels")
print(f"Val   : {vi:5d} images, {vl:5d} labels")
print(f"Test  : {sti:5d} images, {stl:5d} labels")

assert ti   >= 100, f"Too few training images: {ti}"
assert vi   >= 20,  f"Too few val images: {vi}"
assert DATA_YAML.exists(), "data.yaml missing"
print("\nDataset verification passed.")

## 10. Data Integrity Audit

In [ ]:
train_img_dir = YOLO_ROOT / "images" / "train"
train_lbl_dir = YOLO_ROOT / "labels" / "train"
all_train = list(train_img_dir.glob("*.jpg"))

sample = random.sample(all_train, min(500, len(all_train)))
corrupt, widths, heights, empty_lbl = 0, [], [], 0

for img_path in sample:
    img = cv2.imread(str(img_path))
    if img is None:
        corrupt += 1
        continue
    h, w = img.shape[:2]
    widths.append(w)
    heights.append(h)
    lbl_path = train_lbl_dir / img_path.with_suffix(".txt").name
    if lbl_path.exists() and lbl_path.stat().st_size == 0:
        empty_lbl += 1

print(f"Sampled      : {len(sample)}")
print(f"Corrupt      : {corrupt}")
print(f"Empty labels : {empty_lbl}")
if widths:
    print(f"Width range  : {min(widths)}–{max(widths)} px")
    print(f"Height range : {min(heights)}–{max(heights)} px")

## 11. Label/Schema Inspection

In [ ]:
all_lbls = list(train_lbl_dir.glob("*.txt"))
total_boxes = 0
box_areas   = []
invalid     = 0

for lbl_path in all_lbls:
    for line in lbl_path.read_text().strip().splitlines():
        parts = line.strip().split()
        if len(parts) != 5:
            invalid += 1
            continue
        w, h = float(parts[3]), float(parts[4])
        box_areas.append(w * h)
        total_boxes += 1

print(f"Total face boxes : {total_boxes:,}")
print(f"Invalid lines    : {invalid}")
if box_areas:
    small = sum(1 for a in box_areas if a < 0.001)
    print(f"Tiny faces (<0.1% img area) : {small:,} ({100*small/len(box_areas):.1f}%)")
    print(f"BBox area (WxH norm)  min={min(box_areas):.5f}  mean={sum(box_areas)/len(box_areas):.5f}  max={max(box_areas):.5f}")

In [ ]:
## 12. Sample Visualization

def draw_yolo_boxes(img: np.ndarray, lbl_path: Path, class_names: list[str]) -> np.ndarray:
    out = img.copy()
    h, w = out.shape[:2]
    if not lbl_path.exists():
        return out
    for line in lbl_path.read_text().strip().splitlines():
        parts = line.strip().split()
        if len(parts) != 5:
            continue
        cls_id = int(parts[0])
        xc, yc, bw, bh = map(float, parts[1:])
        x1 = int((xc - bw/2) * w)
        y1 = int((yc - bh/2) * h)
        x2 = int((xc + bw/2) * w)
        y2 = int((yc + bh/2) * h)
        label = class_names[cls_id] if cls_id < len(class_names) else str(cls_id)
        cv2.rectangle(out, (x1, y1), (x2, y2), (0, 255, 100), 2)
        cv2.putText(out, label, (x1, max(y1-4, 10)), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 255, 100), 1)
    return out

class_names = ["face"]
sample_6 = random.sample(all_train, min(6, len(all_train)))

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()
for ax, img_path in zip(axes, sample_6):
    img_bgr = cv2.imread(str(img_path))
    lbl_path = train_lbl_dir / img_path.with_suffix(".txt").name
    n_faces  = sum(1 for l in lbl_path.read_text().strip().splitlines() if l.strip()) if lbl_path.exists() else 0
    vis = draw_yolo_boxes(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB), lbl_path, class_names)
    ax.imshow(cv2.resize(vis, (320, 240)))
    ax.set_title(f"{n_faces} face(s)", fontsize=8)
    ax.axis("off")
plt.suptitle("Sample WIDER FACE training images with GT bboxes", fontsize=12)
plt.tight_layout()
fig.savefig(str(ARTIFACTS_DIR / "sample_images.png"), dpi=100)
plt.close(fig)
print("Saved sample_images.png")

## 13. Preprocessing / Augmentation Strategy

YOLO26m applies augmentation (mosaic, flip, HSV jitter, scale) internally during training.
Below we show a manual Albumentations preview for transparency.

In [ ]:
import albumentations as A

aug = A.Compose([
    A.RandomBrightnessContrast(p=0.5),
    A.HueSaturationValue(p=0.4),
    A.HorizontalFlip(p=0.5),
    A.Rotate(limit=10, p=0.3),
], bbox_params=A.BboxParams(format="yolo", label_fields=["class_labels"]))

sample_img_path = random.choice(all_train)
sample_lbl_path = train_lbl_dir / sample_img_path.with_suffix(".txt").name
img_rgb = cv2.cvtColor(cv2.imread(str(sample_img_path)), cv2.COLOR_BGR2RGB)

bboxes, class_labels = [], []
if sample_lbl_path.exists():
    for line in sample_lbl_path.read_text().strip().splitlines():
        p = line.split()
        if len(p) == 5:
            class_labels.append(int(p[0]))
            bboxes.append([float(x) for x in p[1:]])

try:
    aug_result = aug(image=img_rgb, bboxes=bboxes, class_labels=class_labels)
    aug_img = aug_result["image"]
except Exception:
    aug_img = img_rgb

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(cv2.resize(img_rgb, (320, 240)))
axes[0].set_title("Original")
axes[0].axis("off")
axes[1].imshow(cv2.resize(aug_img, (320, 240)))
axes[1].set_title("Augmented (demo)")
axes[1].axis("off")
plt.tight_layout()
fig.savefig(str(ARTIFACTS_DIR / "augmentation_preview.png"), dpi=100)
plt.close(fig)
print("Saved augmentation_preview.png")

## 14. Train/Validation/Test Split Verification

In [ ]:
split_df = pd.DataFrame({
    "split":  ["train", "val", "test"],
    "images": [ti, vi, sti],
    "labels": [tl, vl, stl],
})
print(split_df.to_string(index=False))
assert ti >= 100, "Insufficient training data"
print("\nSplit verification passed.")

## 15. Baseline or Sanity-Check Pipeline

Run pretrained YOLO26m (COCO weights, which includes `person` class) on a few val images
to verify the pipeline is functional before fine-tuning with face-specific weights.

In [ ]:
from ultralytics import YOLO

sanity_model = YOLO("yolo26m.pt")
val_img_dir  = YOLO_ROOT / "images" / "val"
val_imgs     = list(val_img_dir.glob("*.jpg"))
sanity_imgs  = random.sample(val_imgs, min(6, len(val_imgs)))
val_lbl_dir  = YOLO_ROOT / "labels" / "val"

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()
for ax, img_path in zip(axes, sanity_imgs):
    result = sanity_model.predict(str(img_path), verbose=False)[0]
    n_det  = len(result.boxes) if result.boxes is not None else 0
    lbl_path = val_lbl_dir / img_path.with_suffix(".txt").name
    n_gt   = sum(1 for l in lbl_path.read_text().strip().splitlines() if l.strip()) if lbl_path.exists() else 0
    ax.imshow(result.plot()[:, :, ::-1])
    ax.set_title(f"Pretrained: {n_det} det | GT: {n_gt} face(s)", fontsize=7)
    ax.axis("off")
plt.suptitle("Sanity check — pretrained YOLO26m (COCO)", fontsize=11)
plt.tight_layout()
fig.savefig(str(ARTIFACTS_DIR / "sanity_inference.png"), dpi=100)
plt.close(fig)
print("Saved sanity_inference.png")

## 16. Main Model Setup

In [ ]:
PREFERRED_MODEL = "yolo26m.pt"
FALLBACK_MODEL  = "yolo26s.pt"

IMG_SIZE = 640
EPOCHS   = 30
BATCH    = 16
WORKERS  = 2

TRAIN_PROJECT = RUNS_DIR / "face_det"
TRAIN_NAME    = "yolo26m_widerface"

print(f"Model   : {PREFERRED_MODEL}")
print(f"Epochs  : {EPOCHS}")
print(f"Batch   : {BATCH}")
print(f"Img size: {IMG_SIZE}")
print(f"Data    : {DATA_YAML}")

## 17. Training Loop / Trainer Setup

In [ ]:
from ultralytics import YOLO


def run_training(model_name: str, batch: int) -> object:
    model = YOLO(model_name)
    return model.train(
        data=str(DATA_YAML),
        epochs=EPOCHS,
        imgsz=IMG_SIZE,
        batch=batch,
        workers=WORKERS,
        project=str(TRAIN_PROJECT),
        name=TRAIN_NAME,
        exist_ok=True,
        device=DEVICE,
        seed=SEED,
        verbose=True,
    )


train_model_name = PREFERRED_MODEL
current_batch    = BATCH
try:
    train_results = run_training(train_model_name, current_batch)
    print(f"Training complete with {train_model_name}.")
except RuntimeError as exc:
    if "out of memory" in str(exc).lower():
        print(f"OOM with {train_model_name}, retrying with {FALLBACK_MODEL}...")
        torch.cuda.empty_cache()
        train_model_name = FALLBACK_MODEL
        current_batch    = max(4, current_batch // 2)
        train_results    = run_training(train_model_name, current_batch)
        print(f"Training complete with {train_model_name} (OOM fallback).")
    else:
        raise

## 18. Validation and Core Metrics

In [ ]:
from ultralytics import YOLO
import json

best_path = TRAIN_PROJECT / TRAIN_NAME / "weights" / "best.pt"
if not best_path.exists():
    raise FileNotFoundError(f"best.pt not found at {best_path}")

best_model  = YOLO(str(best_path))
val_metrics = best_model.val(
    data=str(DATA_YAML),
    split="val",
    imgsz=IMG_SIZE,
    device=DEVICE,
    verbose=True,
)

map50     = float(val_metrics.box.map50)
map50_95  = float(val_metrics.box.map)
precision = float(val_metrics.box.mp)
recall    = float(val_metrics.box.mr)

print(f"mAP50     : {map50:.4f}")
print(f"mAP50-95  : {map50_95:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")

names     = val_metrics.names
class_aps = {}
ap50_vals = val_metrics.box.ap50
cls_list  = list(names.values()) if isinstance(names, dict) else names
for cls_name, ap in zip(cls_list, ap50_vals):
    class_aps[cls_name] = float(ap)
    print(f"  AP50 [{cls_name}]: {ap:.4f}")

metrics_out = {
    "map50": map50, "map50_95": map50_95,
    "precision": precision, "recall": recall,
    "per_class_ap50": class_aps,
}
with open(ARTIFACTS_DIR / "metrics.json", "w") as f:
    json.dump(metrics_out, f, indent=2)
print("Saved metrics.json")

## 19. Error Analysis

Display training curves and analyse missed faces (false negatives) — a key failure mode for tiny/occluded faces.

In [ ]:
run_dir = TRAIN_PROJECT / TRAIN_NAME

for fname in ["results.png", "PR_curve.png", "F1_curve.png", "confusion_matrix.png"]:
    fpath = run_dir / fname
    if fpath.exists():
        img = cv2.cvtColor(cv2.imread(str(fpath)), cv2.COLOR_BGR2RGB)
        fig, ax = plt.subplots(figsize=(10, 5))
        ax.imshow(img)
        ax.axis("off")
        ax.set_title(fname)
        plt.tight_layout()
        fig.savefig(str(ARTIFACTS_DIR / fname), dpi=80)
        plt.close(fig)
        print(f"Saved {fname}")

# False negative analysis: images where model returns 0 faces but GT ≥1
val_all_imgs = list((YOLO_ROOT / "images" / "val").glob("*.jpg"))
val_lbl_dir  = YOLO_ROOT / "labels" / "val"
fn_examples  = []

for img_path in random.sample(val_all_imgs, min(300, len(val_all_imgs))):
    lbl_path = val_lbl_dir / img_path.with_suffix(".txt").name
    if not lbl_path.exists() or lbl_path.stat().st_size == 0:
        continue
    result = best_model.predict(str(img_path), conf=0.25, verbose=False)[0]
    n_det  = len(result.boxes) if result.boxes is not None else 0
    if n_det == 0:
        fn_examples.append(img_path)
    if len(fn_examples) >= 6:
        break

if fn_examples:
    fig, axes = plt.subplots(2, 3, figsize=(14, 8))
    axes = axes.flatten()
    for ax, img_path in zip(axes, fn_examples):
        img_rgb  = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
        lbl_path = val_lbl_dir / img_path.with_suffix(".txt").name
        n_gt = sum(1 for l in lbl_path.read_text().strip().splitlines() if l.strip())
        vis  = draw_yolo_boxes(img_rgb, lbl_path, class_names)
        ax.imshow(cv2.resize(vis, (320, 240)))
        ax.set_title(f"GT: {n_gt} face(s)\nPred: 0 detections", fontsize=8, color="red")
        ax.axis("off")
    plt.suptitle("False negatives — missed faces", fontsize=11)
    plt.tight_layout()
    fig.savefig(str(ARTIFACTS_DIR / "false_negatives.png"), dpi=100)
    plt.close(fig)
    print(f"Saved false_negatives.png ({len(fn_examples)} examples)")
else:
    print("No false negative examples found in sampled val set.")

## 20. Inference on Holdout Examples

In [ ]:
test_img_dir  = YOLO_ROOT / "images" / "test"
test_all_imgs = list(test_img_dir.glob("*.jpg"))
if not test_all_imgs:
    test_all_imgs = val_all_imgs

holdout_sample = random.sample(test_all_imgs, min(6, len(test_all_imgs)))

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()
for ax, img_path in zip(axes, holdout_sample):
    result = best_model.predict(str(img_path), conf=0.25, verbose=False)[0]
    n_det  = len(result.boxes) if result.boxes is not None else 0
    ax.imshow(result.plot()[:, :, ::-1])
    ax.set_title(f"{img_path.name[:22]}\n{n_det} face(s)", fontsize=7)
    ax.axis("off")
plt.suptitle("Holdout inference results", fontsize=11)
plt.tight_layout()
fig.savefig(str(ARTIFACTS_DIR / "holdout_inference.png"), dpi=100)
plt.close(fig)
print("Saved holdout_inference.png")

## 21. Save Model/Artifacts

In [ ]:
import json, shutil
from ultralytics import YOLO

saved_best_pt = ARTIFACTS_DIR / "best.pt"
shutil.copy2(best_path, saved_best_pt)
print(f"Saved best.pt → {saved_best_pt}")

export_model = YOLO(str(saved_best_pt))
onnx_path = export_model.export(format="onnx", imgsz=IMG_SIZE, opset=12)
print(f"Exported ONNX → {onnx_path}")

with open(ARTIFACTS_DIR / "metrics.json") as f:
    m = json.load(f)

manifest = {
    "project":    "Real Time Face Detector (YOLO26m)",
    "model_file": "best.pt",
    "onnx_file":  "best.onnx",
    "dataset":    "mksaad/wider-face-a-face-detection-benchmark",
    "metrics":    m,
}
with open(ARTIFACTS_DIR / "artifact_manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)
print("Saved artifact_manifest.json")

## 22. Reproducibility Notes

- Random seed fixed to `42` throughout
- YOLO26m training uses `seed=42`
- WIDER FACE conversion capped at 8,000 train + 2,000 val for tractable training
- `kagglehub` caches downloads; re-running reuses the cache
- GPU hardware differences may cause minor metric variation

## 23. Conclusion and Limitations

This notebook implements a real end-to-end face detection pipeline on WIDER FACE:
- Real Kaggle download with fail-loud credential checks
- MATLAB `.mat` → YOLO annotation conversion with bbox clamping
- YOLO26m fine-tuned with OOM fallback to YOLO26s
- mAP50, mAP50-95, precision, recall, per-class AP50 + metrics.json
- False negative analysis (missed faces) + holdout inference grid

**Limitations:**
- Capped at 8,000 training images; full WIDER FACE has ~12,880 train images — more data improves recall on tiny/hard faces
- Tiny faces (height < 10px) are difficult for any single-stage detector; WiderFace-tier performance requires specialised anchors or multi-scale heads
- WIDER FACE "Hard" split has extremely occluded faces — expect lower recall there even after full training